# Iraq Farms Health — WorldCereal crop-type pilot

Runs **ESA WorldCereal's own open-source model** (default pre-trained model, which separates **wheat** and **barley**) for one small test area in Iraq, on the Copernicus Data Space cloud — and gives you the result file to send back or put on GitHub.

**Test area:** ~16 × 16 km (≈250 km²) around FTW-FIWF1K near Arab Dulaim — `43.975, 33.870 → 44.146, 34.013`
**Season:** the 2026 growing year; WorldCereal picks the exact window from its crop calendar for this location.
**Cost:** roughly **40 openEO credits** (your account gets 10,000 free per month). **Time:** ~20–40 min of cloud processing.

**How to use:** run the cells **top to bottom** (▶ on each, or *Runtime → Run all*). One cell asks you to log in to Copernicus.

Code: [WorldCereal/worldcereal-classification](https://github.com/WorldCereal/worldcereal-classification) pinned at commit `b739085` (MIT licence), so this run is reproducible.

## 1 · Settings
Change nothing for the pilot. `SAVE_TO_DRIVE = True` keeps the results (and the job's progress) in your Google Drive, so a Colab disconnect doesn't lose anything — just re-run the cells and it resumes.

In [ ]:
# ---- Settings -------------------------------------------------------------
AREA_NAME = "pilot_arab_dulaim"                  # becomes the folder / file name
BBOX      = (43.975, 33.870, 44.146, 34.013)      # west, south, east, north (degrees)
YEAR      = 2026                                  # growing year to map
SAVE_TO_DRIVE = True                              # keep results in Google Drive
# ---------------------------------------------------------------------------

import json, os
W, S, E, N = BBOX
grid = {"type": "FeatureCollection", "features": [{
    "type": "Feature", "properties": {"tile_name": AREA_NAME},
    "geometry": {"type": "Polygon", "coordinates": [[[W,S],[E,S],[E,N],[W,N],[W,S]]]}}]}
os.makedirs("/content/aoi", exist_ok=True)
GRID_PATH = f"/content/aoi/{AREA_NAME}.geojson"
with open(GRID_PATH, "w") as fh:
    json.dump(grid, fh)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/iraq-farms-health/worldcereal/{AREA_NAME}_{YEAR}"
else:
    OUT = f"/content/worldcereal_out/{AREA_NAME}_{YEAR}"
os.makedirs(OUT, exist_ok=True)

import shlex, subprocess
def sh(cmd):
    # Run a shell command, streaming its output live (unbuffered) into the notebook.
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {p.returncode}) — see the messages above.")

import math
km2 = abs(E-W) * 111.32 * math.cos(math.radians((S+N)/2)) * abs(N-S) * 110.57
print(f"Area: {AREA_NAME}  ≈ {km2:.0f} km²  (WorldCereal limit per run: 2,500 km²)")
print("Results will be saved in:", OUT)

## 2 · Install WorldCereal (≈3–8 min)
Installed into its **own environment** (`/content/wcenv`) so it can't clash with Colab's built-in packages — no runtime restart needed.

In [ ]:
%%bash
set -e
pip -q install uv
if [ ! -d /content/worldcereal-classification ]; then
  git clone -q --filter=blob:none https://github.com/WorldCereal/worldcereal-classification.git /content/worldcereal-classification
fi
cd /content/worldcereal-classification && git checkout -q b739085 && git log -1 --format='WorldCereal code: %h  %ad  %s' --date=short
if [ ! -x /content/wcenv/bin/python ]; then uv venv -q -p 3.12 /content/wcenv; fi
VIRTUAL_ENV=/content/wcenv uv pip install -q /content/worldcereal-classification rasterio matplotlib
/content/wcenv/bin/python -c "import worldcereal, openeo; print('Installed OK — openeo', openeo.__version__)"

## 3 · Log in to Copernicus Data Space
Copernicus doesn't use an API key for this — it uses a **login link**. Run the cell, click the link it prints, sign in with your Copernicus account (the one you used on the WorldCereal hub), approve, and come back. The cell finishes by itself once you've approved (you have 15 minutes).

*Optional — no clicking:* if you have Copernicus **OAuth client credentials**, add them under 🔑 **Secrets** (left sidebar) as `CDSE_CLIENT_ID` and `CDSE_CLIENT_SECRET` and enable notebook access; this cell will use them instead.

In [ ]:
import os, subprocess, textwrap
try:
    from google.colab import userdata
    cid, sec = userdata.get("CDSE_CLIENT_ID"), userdata.get("CDSE_CLIENT_SECRET")
    os.environ.update({"OPENEO_AUTH_METHOD": "client_credentials",
                       "OPENEO_AUTH_CLIENT_ID_CDSE": cid, "OPENEO_AUTH_CLIENT_SECRET_CDSE": sec,
                       "OPENEO_AUTH_PROVIDER_ID_CDSE": "CDSE"})
    print("Using client credentials from Colab Secrets.")
except Exception:
    os.environ.pop("OPENEO_AUTH_METHOD", None)
    print("No client credentials in Secrets — using the login link.")

login = textwrap.dedent('''
    import os, openeo
    c = openeo.connect("openeo.dataspace.copernicus.eu")
    if os.environ.get("OPENEO_AUTH_METHOD") == "client_credentials":
        c.authenticate_oidc_client_credentials(
            client_id=os.environ["OPENEO_AUTH_CLIENT_ID_CDSE"],
            client_secret=os.environ["OPENEO_AUTH_CLIENT_SECRET_CDSE"],
            provider_id=os.environ.get("OPENEO_AUTH_PROVIDER_ID_CDSE"))
    else:
        # Saves a refresh token, so the run in step 4 logs in without asking again.
        c.authenticate_oidc_device(max_poll_time=900, store_refresh_token=True)
    me = c.describe_account()
    print("Logged in as:", me.get("name") or me.get("user_id"))
''')
with open("/content/cdse_login.py", "w") as fh:
    fh.write(login)
sh("/content/wcenv/bin/python -u /content/cdse_login.py")

## 4 · Run the crop-type map (≈20–40 min)
This submits one job to the Copernicus cloud and waits for it, printing a status line every minute, then downloads the result into your output folder.

- **If Colab disconnects:** reconnect, run steps 1–3 again, then this cell — it picks up the same job from the tracking file in Drive instead of starting (and paying for) a new one.
- **If you stop it on purpose:** the cloud job keeps running. Cancel it at [openeo.dataspace.copernicus.eu](https://openeo.dataspace.copernicus.eu/) if you don't want it to finish.

In [ ]:
args = [
    "--task", "classification",
    "--grid_path", GRID_PATH, "--grid_size", "none",
    "--year", str(YEAR),
    "--product", "croptype",
    "--enable-cropland-head", "--enable-croptype-head", "--mask-cropland",
    "--enable-cropland-postprocess", "--cropland-postprocess-method", "majority_vote", "--cropland-postprocess-kernel-size", "3",
    "--enable-croptype-postprocess", "--croptype-postprocess-method", "majority_vote", "--croptype-postprocess-kernel-size", "5",
    "--class-probabilities", "--merge-classification-products",
    "--parallel_jobs", "1", "--restart_failed",
    "--output_folder", OUT,
]
sh("cd /content/worldcereal-classification && /content/wcenv/bin/python -u scripts/inference/run_worldcereal_task_openeo.py "
   + " ".join(shlex.quote(a) for a in args))

## 5 · Check the result
Lists what came back, the crop classes WorldCereal found (wheat and barley flagged), the hectares of each, the credits used, and draws a preview. Also writes `summary.json`.

In [ ]:
%%writefile /content/inspect_results.py
"""Summarise a WorldCereal crop-type run: legend, class shares, preview, cost.

Usage: python inspect_results.py <output_folder>
Reads the GeoTIFF(s) written by run_worldcereal_task_openeo.py, recovers the
class legend from the per-class probability band names
("croptype_probability:<season>:<class>", in class-index order), and writes
summary.json + preview_<season>.png next to the outputs.
"""
import csv
import glob
import json
import os
import sys

import numpy as np
import rasterio

NOCROP = 254          # WorldCereal: pixel outside the cropland mask
NODATA = {255, 65535}

out_dir = sys.argv[1]
tifs = sorted(glob.glob(os.path.join(out_dir, "**", "*.tif"), recursive=True))
if not tifs:
    sys.exit(f"No GeoTIFFs found under {out_dir} — did the job finish and download?")

summary = {"files": [], "seasons": {}, "cost": None}
for tif in tifs:
    with rasterio.open(tif) as ds:
        names = [d or f"band{i+1}" for i, d in enumerate(ds.descriptions)]
        summary["files"].append({"file": os.path.relpath(tif, out_dir), "bands": names,
                                 "size_px": [ds.width, ds.height], "crs": str(ds.crs),
                                 "pixel_m": round(abs(ds.transform.a), 2)})
        print(f"\n{os.path.relpath(tif, out_dir)}  {ds.width}x{ds.height} px, {ds.crs}, {len(names)} bands")
        for n in names:
            print("   ", n)

        # Legend per season from the per-class probability band order.
        legend = {}
        for n in names:
            parts = n.split(":")
            if len(parts) == 3 and parts[0] == "croptype_probability":
                legend.setdefault(parts[1], []).append(parts[2])

        for i, n in enumerate(names):
            if not n.startswith("croptype_classification:"):
                continue
            season = n.split(":", 1)[1]
            classes = legend.get(season) or next(iter(legend.values()), [])
            arr = ds.read(i + 1)
            valid = ~np.isin(arr, list(NODATA))
            total = int(valid.sum())
            nocrop = int((arr == NOCROP).sum())
            vals, counts = np.unique(arr[valid & (arr != NOCROP)], return_counts=True)
            px_ha = (abs(ds.transform.a) * abs(ds.transform.e)) / 10000.0
            rows = []
            for v, c in sorted(zip(vals.tolist(), counts.tolist()), key=lambda t: -t[1]):
                name = classes[v] if v < len(classes) else f"class_{v}"
                rows.append({"code": v, "class": name, "pixels": c,
                             "hectares": round(c * px_ha, 1),
                             "share_of_cropland_pct": round(100 * c / max(1, total - nocrop), 1)})
            summary["seasons"][season] = {"legend": classes, "valid_pixels": total,
                                          "not_cropland_pct": round(100 * nocrop / max(1, total), 1),
                                          "classes": rows}
            print(f"\nSeason {season}: {round(100*nocrop/max(1,total),1)}% of the area is not cropland. Cropland by class:")
            for r in rows:
                flag = "  <==" if r["class"] in ("wheat", "barley") else ""
                print(f"   {r['class']:<22}{r['hectares']:>10} ha  {r['share_of_cropland_pct']:>5}%{flag}")

            # Preview: wheat and barley in strong colours, other crops muted, non-crop pale.
            try:
                import matplotlib
                matplotlib.use("Agg")
                import matplotlib.pyplot as plt
                from matplotlib.colors import ListedColormap
                from matplotlib.patches import Patch
                img = np.full(arr.shape, 0, dtype=np.uint8)          # 0 = no data
                img[arr == NOCROP] = 1                                # not cropland
                img[valid & (arr != NOCROP)] = 2                      # other crop
                for v, name in enumerate(classes):
                    if name == "wheat":
                        img[arr == v] = 3
                    elif name == "barley":
                        img[arr == v] = 4
                cmap = ListedColormap(["#ffffff", "#ece6d6", "#9fb7a8", "#d9a441", "#1e6e5a"])
                fig, ax = plt.subplots(figsize=(8, 8), dpi=120)
                ax.imshow(img, cmap=cmap, vmin=0, vmax=4, interpolation="nearest")
                ax.set_axis_off()
                ax.set_title(f"WorldCereal crop type — season {season}")
                ax.legend(handles=[Patch(color="#d9a441", label="wheat"), Patch(color="#1e6e5a", label="barley"),
                                   Patch(color="#9fb7a8", label="other crops"), Patch(color="#ece6d6", label="not cropland")],
                          loc="lower right", framealpha=.9)
                png = os.path.join(os.path.dirname(tif), f"preview_{season}.png")
                fig.savefig(png, bbox_inches="tight")
                plt.close(fig)
                print(f"   preview: {os.path.relpath(png, out_dir)}")
            except Exception as e:  # preview is a convenience, never a failure
                print("   (preview skipped:", e, ")")

# Cost: the job manager's tracking file and per-job metadata.
track = os.path.join(out_dir, "job_tracking.csv")
if os.path.exists(track):
    with open(track, newline="") as fh:
        rows = list(csv.DictReader(fh))
    summary["jobs"] = [{k: r.get(k) for k in ("tile_name", "id", "status", "costs", "duration", "cpu", "memory") if k in r}
                       for r in rows]
    costs = [float(r["costs"]) for r in rows if r.get("costs") not in (None, "", "nan")]
    if costs:
        summary["cost"] = round(sum(costs), 2)
    print("\nJobs:", json.dumps(summary["jobs"], indent=1))
print(f"\nCredits used: {summary['cost'] if summary['cost'] is not None else 'see the job JSON / CDSE dashboard'}")

with open(os.path.join(out_dir, "summary.json"), "w") as fh:
    json.dump(summary, fh, indent=1)
print("Wrote summary.json")

In [ ]:
sh(f"/content/wcenv/bin/python -u /content/inspect_results.py {shlex.quote(OUT)}")
import glob
from IPython.display import Image, display
for png in sorted(glob.glob(f"{OUT}/**/preview_*.png", recursive=True)):
    display(Image(png, width=600))

## 6 · Package the result
Zips the whole output folder (GeoTIFF, job metadata, cost, summary, preview) into one file and downloads it. Send that file back in our chat — or attach it to a GitHub release (GitHub allows files up to 2 GB in releases; plain repo files are limited to 100 MB).

In [ ]:
import shutil, os
zip_base = f"/content/worldcereal_{os.path.basename(OUT)}"
zip_path = shutil.make_archive(zip_base, "zip", OUT)
print(f"{zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
if SAVE_TO_DRIVE:
    shutil.copy(zip_path, os.path.dirname(OUT))
    print("Also saved to Drive:", os.path.join(os.path.dirname(OUT), os.path.basename(zip_path)))
from google.colab import files
files.download(zip_path)